# cell 1: import

In [12]:
import pandas as pd
import re
from pathlib import Path

In [13]:
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")

INPUT_PATH = BASE_DIR / "data_outputs" / "step01_clean_text" / "01_cv_cleaned.xlsx"
OUTPUT_DIR = BASE_DIR / "data_outputs" / "step02_sections"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "02_cv_sectioned.xlsx"

print("Input:", INPUT_PATH)
print("Exists:", INPUT_PATH.exists())
print("Output:", OUTPUT_PATH)

Input: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step01_clean_text/01_cv_cleaned.xlsx
Exists: True
Output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step02_sections/02_cv_sectioned.xlsx


In [14]:
df = pd.read_excel(INPUT_PATH, engine="openpyxl")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (20, 6)
Columns: ['candidate_id', 'cv_text_raw', 'cv_text_clean', 'raw_length', 'clean_length', 'is_empty_clean']


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False
2,C003,"Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...,163,154,False
3,C004,"DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...,173,165,False
4,C005,"QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...,172,166,False


In [15]:
required_cols = ["candidate_id", "cv_text_clean"]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_cols}")

print("Đủ cột để tách section.")

Đủ cột để tách section.


In [16]:
SECTION_PATTERNS = {
    "summary": [
        r"\bsummary\b",
        r"\bprofile\b",
        r"\bprofessional summary\b",
        r"\bobjective\b",
        r"\bcareer objective\b",
        r"\babout me\b"
    ],
    "skills": [
        r"\bskills\b",
        r"\btechnical skills\b",
        r"\bcore skills\b",
        r"\bkey skills\b",
        r"\bcompetencies\b"
    ],
    "experience": [
        r"\bexperience\b",
        r"\bwork experience\b",
        r"\bemployment history\b",
        r"\bprofessional experience\b"
    ],
    "education": [
        r"\beducation\b",
        r"\bacademic background\b",
        r"\bqualification\b"
    ],
    "projects": [
        r"\bprojects\b",
        r"\bpersonal projects\b",
        r"\bproject experience\b"
    ],
    "certifications": [
        r"\bcertifications\b",
        r"\bcertificates\b",
        r"\blicenses\b"
    ]
}

In [17]:
def find_section_positions(text, section_patterns):
    matches = []

    for section_name, patterns in section_patterns.items():
        for pattern in patterns:
            for match in re.finditer(pattern, text, flags=re.IGNORECASE):
                matches.append({
                    "section": section_name,
                    "start": match.start(),
                    "end": match.end(),
                    "matched_text": match.group()
                })

    matches = sorted(matches, key=lambda x: x["start"])
    return matches

In [18]:
def split_cv_sections(text: str) -> dict:
    empty_result = {
        "section_summary": "",
        "section_skills": "",
        "section_experience": "",
        "section_education": "",
        "section_projects": "",
        "section_certifications": "",
        "section_other": ""
    }

    if pd.isna(text) or not str(text).strip():
        return empty_result

    text = str(text)
    matches = find_section_positions(text, SECTION_PATTERNS)

    # nếu không tìm thấy heading nào thì nhét toàn bộ vào other
    if not matches:
        empty_result["section_other"] = text
        return empty_result

    result = empty_result.copy()

    for i, match in enumerate(matches):
        section_name = match["section"]
        start_content = match["end"]

        if i < len(matches) - 1:
            end_content = matches[i + 1]["start"]
        else:
            end_content = len(text)

        content = text[start_content:end_content].strip(" :-|")
        key = f"section_{section_name}"

        if key in result:
            if result[key]:
                result[key] += " " + content
            else:
                result[key] = content

    # phần text trước heading đầu tiên
    first_start = matches[0]["start"]
    if first_start > 0:
        result["section_other"] = text[:first_start].strip()

    return result

In [19]:
section_data = df["cv_text_clean"].apply(split_cv_sections)
section_df = pd.DataFrame(section_data.tolist())

df_out = pd.concat([df, section_df], axis=1)

display(df_out.head())

,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,section_projects,section_certifications,section_other
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,,,,,,,frontend developer experienced in reactjs vuej...
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,,,,,,,backend engineer experienced in java spring bo...
2,C003,"Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...,163,154,False,,,,,,,data analyst experienced in python pandas nump...
3,C004,"DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...,173,165,False,,,,,,,devops engineer experienced in linux docker je...
4,C005,"QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...,172,166,False,,,,,,,qa engineer experienced in manual testing test...


In [20]:
section_cols = [
    "section_summary",
    "section_skills",
    "section_experience",
    "section_education",
    "section_projects",
    "section_certifications",
    "section_other"
]

for col in section_cols:
    df_out[f"{col}_len"] = df_out[col].fillna("").apply(len)

display(df_out[["candidate_id"] + [f"{col}_len" for col in section_cols]].head())

,candidate_id,section_summary_len,section_skills_len,section_experience_len,section_education_len,section_projects_len,section_certifications_len,section_other_len
0,C001,0,0,0,0,0,0,240
1,C002,0,0,0,0,0,0,181
2,C003,0,0,0,0,0,0,154
3,C004,0,0,0,0,0,0,165
4,C005,0,0,0,0,0,0,166


In [21]:
df_out.to_excel(OUTPUT_PATH, index=False)
print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step02_sections/02_cv_sectioned.xlsx


In [22]:
result = pd.read_excel(OUTPUT_PATH, engine="openpyxl")
print("Output shape:", result.shape)
display(result.head())

Output shape: (20, 20)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,section_projects,section_certifications,section_other,section_summary_len,section_skills_len,section_experience_len,section_education_len,section_projects_len,section_certifications_len,section_other_len
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,NaN,NaN,NaN,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...,0,0,0,0,0,0,240
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,NaN,NaN,NaN,NaN,NaN,NaN,backend engineer experienced in java spring bo...,0,0,0,0,0,0,181
2,C003,"Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...,163,154,False,NaN,NaN,NaN,NaN,NaN,NaN,data analyst experienced in python pandas nump...,0,0,0,0,0,0,154
3,C004,"DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...,173,165,False,NaN,NaN,NaN,NaN,NaN,NaN,devops engineer experienced in linux docker je...,0,0,0,0,0,0,165
4,C005,"QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...,172,166,False,NaN,NaN,NaN,NaN,NaN,NaN,qa engineer experienced in manual testing test...,0,0,0,0,0,0,166
